# Brain Tumour Detection and Classification (YOLOv11-seg)

Runtime > Change runtime type > **T4 GPU**, phir cells upar se neeche run karo.

In [ ]:
!pip install -q ultralytics roboflow gradio opencv-python matplotlib onnx

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Imports

In [ ]:
import os
import shutil
from getpass import getpass
from collections import defaultdict

import matplotlib.pyplot as plt
from roboflow import Roboflow
from ultralytics import YOLO

## Download dataset from Roboflow

In [ ]:
# NOTE: API key code me hard-code mat karo. Roboflow > Settings > API Keys se nayi key lo.
api_key = getpass("Roboflow API key: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("deneme-tfvg6").project("tumorneuro")
version = project.version(1)
dataset = version.download("yolov11")   # folder: tumorneuro-1

print(os.listdir("tumorneuro-1"))       # train, valid, test, data.yaml

## Class distribution

In [ ]:
class_names = ['Glioma', 'Meningioma', 'Pituitary tumor']
dataset_base = "tumorneuro-1"
splits = ['train', 'valid', 'test']

split_counts = {s: defaultdict(int) for s in splits}
for split in splits:
    label_dir = os.path.join(dataset_base, split, 'labels')
    if not os.path.exists(label_dir):
        print(f"Warning: {label_dir} not found, skipping")
        continue
    for label_file in os.listdir(label_dir):
        if label_file.endswith('.txt'):
            with open(os.path.join(label_dir, label_file)) as f:
                for line in f:
                    if line.strip():
                        split_counts[split][int(line.split()[0])] += 1

totals = [sum(split_counts[s][i] for s in splits) for i in range(len(class_names))]

plt.figure(figsize=(10, 6))
bars = plt.bar(class_names, totals)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., h, f'{int(h):,}', ha='center', va='bottom')
plt.title("Brain Tumor Class Distribution")
plt.xlabel("Tumor Type")
plt.ylabel("Number of Instances")
plt.grid(axis='y', alpha=0.3)
plt.show()

## Train/Val/Test stacked chart

In [ ]:
split_colors = {'train': '#1f77b4', 'valid': '#ff7f0e', 'test': '#2ca02c'}
fig, ax = plt.subplots(figsize=(12, 7))
for i, name in enumerate(class_names):
    bottom = 0
    for split in splits:
        count = split_counts[split][i]
        ax.bar(name, count, bottom=bottom, color=split_colors[split],
               label=split if i == 0 else "")
        bottom += count
ax.set_title("Class Distribution with Train/Val/Test Split")
ax.set_xlabel("Tumor Type")
ax.set_ylabel("Number of Instances")
ax.legend(title="Split")
plt.show()

## Train

In [ ]:
model = YOLO("yolo11n-seg.pt")
train_results = model.train(
    data="/content/tumorneuro-1/data.yaml",
    epochs=10,      # zyada epochs = better accuracy (e.g. 30-50)
    imgsz=640,
    batch=16,
    device=0,
)

## Training results (plots)

In [ ]:
from IPython.display import Image, display
run_dir = str(model.trainer.save_dir)   # e.g. /content/runs/segment/train
for name in ["results.png", "confusion_matrix_normalized.png", "train_batch0.jpg",
             "MaskF1_curve.png", "MaskPR_curve.png"]:
    p = os.path.join(run_dir, name)
    if os.path.exists(p):
        display(Image(p))

## Test on unseen images

In [ ]:
test_results = model.predict(
    source="tumorneuro-1/test/images/",
    conf=0.25,
    save=True, save_txt=True, save_conf=True,
    show=False, show_labels=True, show_conf=True,
    device=0,
)

## Export ONNX + save weights to Google Drive

In [ ]:
model.export(format="onnx")

out_dir = "/content/drive/MyDrive/Brain Tumour Detection"
os.makedirs(out_dir, exist_ok=True)
shutil.copy(os.path.join(run_dir, "weights", "best.pt"), os.path.join(out_dir, "best.pt"))
shutil.copy(os.path.join(run_dir, "weights", "best.onnx"), os.path.join(out_dir, "best.onnx"))
print("Model saved to Google Drive:", out_dir)

## Gradio web app (link milega)

In [ ]:
from pathlib import Path
model_path = Path(run_dir) / 'weights' / 'best.pt'

import os

import cv2
import gradio as gr
from ultralytics import YOLO

WEIGHTS = str(model_path) if "model_path" in globals() else "best.pt"   # training se mili best.pt
CONF = 0.25

model = YOLO(WEIGHTS)


def detect_tumor(image):
    if image is None:
        return None, [{"Result": "Please upload an image"}]

    image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    results = model.predict(image_bgr, conf=CONF)

    output_image = image
    response = []

    for result in results:
        annotated = result.plot()
        output_image = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                cls_id = int(box.cls[0])
                response.append({
                    "Type": model.names[cls_id],
                    "Confidence": f"{float(box.conf[0]):.2%}",
                    "BBox": [round(x) for x in box.xyxy[0].tolist()],
                })
        else:
            response.append({"Result": "No tumors detected"})

    return output_image, response


examples = [p for p in ["tumorneuro-1/test/images/Image_1.jpg",
                        "tumorneuro-1/test/images/Image_2.jpg"] if os.path.exists(p)]

interface = gr.Interface(
    fn=detect_tumor,
    inputs=gr.Image(label="Upload Brain Scan"),
    outputs=[
        gr.Image(label="Detection Result"),
        gr.JSON(label="Detection Details"),
    ],
    title="Brain Tumor Detection System",
    description="Upload MRI scan to detect tumors (Glioma, Meningioma, Pituitary)",
    examples=[[p] for p in examples] or None,
)

interface.launch(debug=True, share=True)